# Unstructured
- https://unstructured.io/
- https://unstructured-io.github.io/unstructured/index.html
- https://docs.unstructured.io/api-reference/api-services/python-sdk


In [77]:
%pip install "unstructured[all-docs]" unstructured-client watermark


[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [78]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [79]:
from IPython.display import JSON

import json

from unstructured_client import UnstructuredClient
from unstructured_client.models import shared
from unstructured_client.models.errors import SDKError

from unstructured.partition.html import partition_html
from unstructured.partition.pdf import partition_pdf
from unstructured.staging.base import dict_to_elements, elements_to_json

In [80]:
%load_ext watermark

The watermark extension is already loaded. To reload it, use:
  %reload_ext watermark


In [81]:
%watermark --iversions

langchain          : 0.3.15
langchain_community: 0.3.15
unstructured_client: 0.29.0
lxml               : 5.3.0
langchain_core     : 0.3.31
json               : 2.0.9
IPython            : 8.25.0
unstructured       : 0.16.14



In [82]:
import unstructured.partition

help(unstructured.partition)


Help on package unstructured.partition in unstructured:

NAME
    unstructured.partition

PACKAGE CONTENTS
    api
    auto
    common (package)
    csv
    doc
    docx
    email
    epub
    html (package)
    image
    json
    md
    model_init
    msg
    ndjson
    odt
    org
    pdf
    pdf_image (package)
    ppt
    pptx
    rst
    rtf
    strategies
    text
    text_type
    tsv
    utils (package)
    xlsx
    xml

FILE
    /Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/unstructured/partition/__init__.py




In [83]:
from unstructured.partition.pdf import partition_pdf

# Specify the path to your PDF file
filename = "test.pdf"

# Call the partition_pdf function
# Returns a List[Element] present in the pages of the parsed pdf document
elements = partition_pdf(filename)

# Now, elements is a list of all elements present in the pages of the parsed pdf document

In [84]:
elements

In [85]:
len(elements)

3

In [86]:
element_dict = [el.to_dict() for el in elements]
output = json.dumps(element_dict, indent=2)
print(output)

[
  {
    "type": "Title",
    "element_id": "d55973ad90e40c9b8264852170626b34",
    "text": "Prix du bien Localisation",
    "metadata": {
      "coordinates": {
        "points": [
          [
            77.03050400000001,
            89.53459999999995
          ],
          [
            77.03050400000001,
            116.89459999999997
          ],
          [
            142.665728,
            116.89459999999997
          ],
          [
            142.665728,
            89.53459999999995
          ]
        ],
        "system": "PixelSpace",
        "layout_width": 595.2755,
        "layout_height": 841.8898
      },
      "filename": "test.pdf",
      "languages": [
        "eng"
      ],
      "last_modified": "2025-01-23T16:45:40",
      "page_number": 1,
      "filetype": "application/pdf"
    }
  },
  {
    "type": "NarrativeText",
    "element_id": "5b9160e5123da6e2a077964be5846131",
    "text": "Choix de la personne A 368\u20ac Loire sur Rhone",
    "metadata": {
      

In [87]:
unique_types = set()

for item in element_dict:
    unique_types.add(item['type'])

print(unique_types)

{'NarrativeText', 'Title'}


##### We don't see `Table`, table information is not extracted as we expected, lets use different strategy.

### Table extraction from PDF
- Now let’s say that your PDF has tables and let’s say you want to preserve the structure of the tables. 
- You will have to specify the [strategy](https://unstructured-io.github.io/unstructured/best_practices/strategies.html) parameter as `hi_res`. This will use a combination of computer vision and Optical Character Recognition (OCR) to extract the tables and maintain the structure. 
It will return both the text and the html of the table. This is super useful for rendering the tables or passing to a LLM.

> Note: For even better table extraction Unstructured offers an API that improves upon the existing open source models.

> Depending upon machine, you might face different module / library issues, these links might help
- https://stackoverflow.com/questions/59690698/modulenotfounderror-no-module-named-lzma-when-building-python-using-pyenv-on
- https://unstructured-io.github.io/unstructured/installation/full_installation.html

In [88]:
# 1st way

from unstructured.partition.auto import partition

elements = partition(filename=filename,
                     strategy='hi_res',
           )

tables = [el for el in elements if el.category == "Table"]

print(tables[0].text)
print(tables[0].metadata.text_as_html)

INFO: Reading PDF for file: test.pdf ...


Prix du bien Localisation Choix de la personne A 368€ Loire sur Rhone Choix de la personne B 281€ Lyon
None


In [89]:
# 2nd way 

from unstructured.partition.pdf import partition_pdf

elements = partition_pdf(filename=filename,
                         infer_table_structure=True,
                         strategy='hi_res',
           )

tables = [el for el in elements if el.category == "Table"]

print(tables[0].text)
print(tables[0].metadata.text_as_html)

INFO: Reading PDF for file: test.pdf ...


Prix du bien Localisation Choix de la personne A 368€ Loire sur Rhone Choix de la personne B 281€ Lyon
<table><thead><tr><th></th><th>Choix de la personne A</th><th>Choix de la personne B</th></tr></thead><tbody><tr><td>Prix du bien</td><td>368€</td><td>281€</td></tr><tr><td>Localisation</td><td>Loire sur Rhone</td><td>Lyon</td></tr></tbody></table>


### Now, lets use python sdk to do the same -> [link](https://unstructured-io.github.io/unstructured/api.html)
- For SAAS API, visit this [link](https://unstructured.io/api-key-hosted)
- For Free Unstructured API, visit this [link](https://docs.unstructured.io/api-reference/api-services/free-api)

In [90]:
%pip install python-dotenv


[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [91]:
client = UnstructuredClient(
    api_key_auth='awAtYN4DS21HCALStSAmMxTq7HSvQd'
    #api_key_auth=saas_api_key_auth,
    #server_url=saas_server_url,
)

In [92]:
# 3rd way 

with open(filename, "rb") as f:
    files=shared.Files(
        content=f.read(),
        file_name=filename,
    )

req = shared.PartitionParameters(
    files=files,
    strategy="hi_res",
    hi_res_model_name="yolox",
    skip_infer_table_types=[],
    pdf_infer_table_structure=True,
)

try:
    resp = client.general.partition(req)
    elements = dict_to_elements(resp.elements)
except SDKError as e:
    print(e)

TypeError: General.partition() takes 1 positional argument but 2 were given

In [35]:
tables = [el for el in elements if el.category == "Table"]

In [25]:
tables

In [24]:
len(tables)

1

In [23]:
tables[0].text

'Model BoolQ PIQA HellaSwag WinoG. ARC-e ARC-c OBQA Avg. GPT4All-J 6B v1.0* GPT4All-J v1.1-breezy* GPT4All-J v1.2-jazzy* GPT4All-J v1.3-groovy* GPT4All-J Lora 6B* GPT4All LLaMa Lora 7B* GPT4All 13B snoozy* GPT4All Falcon Nous-Hermes (Nous-Research, 2023b) Nous-Hermes2 (Nous-Research, 2023c) Nous-Puffin (Nous-Research, 2023d) Dolly 6B* (Conover et al., 2023a) Dolly 12B* (Conover et al., 2023b) Alpaca 7B* (Taori et al., 2023) Alpaca Lora 7B* (Wang, 2023) GPT-J* 6.7B (Wang and Komatsuzaki, 2021) LLama 7B* (Touvron et al., 2023) LLama 13B* (Touvron et al., 2023) Pythia 6.7B* (Biderman et al., 2023) Pythia 12B* (Biderman et al., 2023) Fastchat T5* (Zheng et al., 2023) Fastchat Vicuña* 7B (Zheng et al., 2023) Fastchat Vicuña 13B* (Zheng et al., 2023) StableVicuña RLHF* (Stability-AI, 2023) StableLM Tuned* (Stability-AI, 2023) StableLM Base* (Stability-AI, 2023) Koala 13B* (Geng et al., 2023) Open Assistant Pythia 12B* Mosaic MPT7B (MosaicML-Team, 2023) Mosaic mpt-instruct (MosaicML-Team, 202

In [30]:
tables[0].metadata

### Now, comes the most interesting part ( utilizing the extracted data in most efficient way)

- It's helpful to have an HTML representation of the table so that you can the information to an LLM while maintaining the table structure.

In [93]:
table_html = tables[0].metadata.text_as_html

In [94]:
table_html

'<table><thead><tr><th></th><th>Choix de la personne A</th><th>Choix de la personne B</th></tr></thead><tbody><tr><td>Prix du bien</td><td>368€</td><td>281€</td></tr><tr><td>Localisation</td><td>Loire sur Rhone</td><td>Lyon</td></tr></tbody></table>'

In [95]:
# view what the HTML in the metadata field looks like

from io import StringIO 
from lxml import etree

parser = etree.XMLParser(remove_blank_text=True)
file_obj = StringIO(table_html)
tree = etree.parse(file_obj, parser)
print(etree.tostring(tree, pretty_print=True).decode())

<table>
  <thead>
    <tr>
      <th/>
      <th>Choix de la personne A</th>
      <th>Choix de la personne B</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Prix du bien</td>
      <td>368&#8364;</td>
      <td>281&#8364;</td>
    </tr>
    <tr>
      <td>Localisation</td>
      <td>Loire sur Rhone</td>
      <td>Lyon</td>
    </tr>
  </tbody>
</table>



In [96]:
# let's display this table

from IPython.core.display import HTML
HTML(table_html)

,Choix de la personne A,Choix de la personne B
Prix du bien,368€,281€
Localisation,Loire sur Rhone,Lyon


#### Now, lets plugin in LangChain to summarize these tables using `Llama3` via `Ollama`
#### [Ollama Playlist](https://www.youtube.com/playlist?list=PLz-qytj7eIWX-bpcRtvkixvo9fuejVr8y)

In [71]:
%pip install langchain langchain_core langchain_community


[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [97]:
from langchain_community.chat_models import ChatOllama
from langchain_core.documents import Document
from langchain.chains.summarize import load_summarize_chain

In [98]:
ChatOllama??

Init signature:
ChatOllama(
    *args: Any,
    name: Optional[str] = None,
    cache: Union[langchain_core.caches.BaseCache, bool, NoneType] = None,
    verbose: bool = <factory>,
    callbacks: Union[list[langchain_core.callbacks.base.BaseCallbackHandler], langchain_core.callbacks.base.BaseCallbackManager, NoneType] = None,
    tags: Optional[list[str]] = None,
    metadata: Optional[dict[str, Any]] = None,
    custom_get_token_ids: Optional[Callable[[str], list[int]]] = None,
    base_url: str = 'http://localhost:11434',
    model: str = 'llama2',
    mirostat: Optional[int] = None,
    mirostat_eta: Optional[float] = None,
    mirostat_tau: Optional[float] = None,
    num_ctx: Optional[int] = None,
    num_gpu: Optional[int] = None,
    num_thread: Optional[int] = None,
    num_predict: Optional[int] = None,
    repeat_last_n: Optional[int] = None,
    repeat_penalty: Optional[float] = None,
    temperature: Optional[float] = None,
    stop: Optional[List[str]] = None,
    tfs_z: O

In [99]:
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

# Prompt personnalisé pour poser une question précise
question = "Quelles sont les deux villes présentes dans ce tableau ?"  # Remplacez par votre question

prompt = PromptTemplate(
    input_variables=["table_html", "question"],
    template="Voici un document HTML : {table_html}. Répondez à la question suivante : {question}"
)

llm = ChatOllama(model="llama3.2")
llm_chain = LLMChain(prompt=prompt, llm=llm)

output = llm_chain.run(table_html=table_html, question=question)

In [100]:
output

'Les deux villes présentes dans ce tableau sont :\n\n1. Loire-sur-Rhône\n2. Lyon'

In [101]:
print(output)

Les deux villes présentes dans ce tableau sont :

1. Loire-sur-Rhône
2. Lyon


#### Convert to pandas df

In [46]:
%pip install pandas


[notice] A new release of pip available: 22.3 -> 24.0
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [59]:
import pandas as pd

# Convert HTML table to pandas DataFrame
dfs = pd.read_html(table_html)

In [54]:
dfs

[                                                Model      BoolQ       PIQA  \
 0                                  GPT4All-J 6B v1.0*       73.4       74.8   
 1                              GPT4All-J v1.1-breezy*         74       75.1   
 2                               GPT4All-J v1.2-jazzy*       74.8       74.9   
 3                              GPT4All-J v1.3-groovy*       73.6        743   
 4                                  GPT4All-J Lora 6B*       68.6       75.8   
 5                              GPT4All LLaMa Lora 7B*       73.1       77.6   
 6                                 GPT4All 13B snoozy*        833       79.2   
 7                                      GPT4All Falcon       77.6       79.8   
 8                  Nous-Hermes (Nous-Research, 2023b)       79.5       78.9   
 9                 Nous-Hermes2 (Nous-Research, 2023c)       83.9       80.7   
 10                 Nous-Puffin (Nous-Research, 2023d)        815       80.7   
 11                  Dolly 6B* (Conover 

In [55]:

# Assuming there's only one table, get the DataFrame
df = dfs[0]

# Now you have the DataFrame
print(df)


                                                Model      BoolQ       PIQA  \
0                                  GPT4All-J 6B v1.0*       73.4       74.8   
1                              GPT4All-J v1.1-breezy*         74       75.1   
2                               GPT4All-J v1.2-jazzy*       74.8       74.9   
3                              GPT4All-J v1.3-groovy*       73.6        743   
4                                  GPT4All-J Lora 6B*       68.6       75.8   
5                              GPT4All LLaMa Lora 7B*       73.1       77.6   
6                                 GPT4All 13B snoozy*        833       79.2   
7                                      GPT4All Falcon       77.6       79.8   
8                  Nous-Hermes (Nous-Research, 2023b)       79.5       78.9   
9                 Nous-Hermes2 (Nous-Research, 2023c)       83.9       80.7   
10                 Nous-Puffin (Nous-Research, 2023d)        815       80.7   
11                  Dolly 6B* (Conover et al., 2023a

In [56]:
df.shape

(38, 9)

In [57]:
df.head()

,Model,BoolQ,PIQA,HellaSwag,WinoG.,ARC-e,ARC-c,OBQA,Avg.
0,GPT4All-J 6B v1.0*,73.4,74.8,63.4,64.7,549,36,40.2,58.2
1,GPT4All-J v1.1-breezy*,74,75.1,63.2,63.6,554,349,38.4,57.8
2,GPT4All-J v1.2-jazzy*,74.8,74.9,63.6,63.8,56.6,353,41,58.6
3,GPT4All-J v1.3-groovy*,73.6,743,63.8,63.5,577,35,38.8,58.1
4,GPT4All-J Lora 6B*,68.6,75.8,66.2,63.5,56.4,357,40.2,58.1
